# 行为校验：先检查错期、权重和联合检验

本 notebook 在主入口最后运行，使用前面的函数；不再次读取大表。校验数据是小型人工面板，不是论文复现结果。这里检查可能改变论文结论的行为，不检查普通排版。

In [ ]:
def test_weight_uses_common_valid_sample():
    sample = pd.DataFrame({monthly_market_value: [100.0, 300.0, 900.0], monthly_excess_ret: [0.02, 0.01, np.nan]})
    assert np.isclose(get_vw_return(sample), 0.0125)
    sample[monthly_excess_ret] = np.nan
    assert np.isnan(get_vw_return(sample))


def test_monthly_alignment_skips_calendar_gaps():
    sample = pd.DataFrame({stock_code: ['000001', '000001'], form: pd.PeriodIndex(['2000-01', '2000-03'], freq='M'), monthly_market_value: [100.0, 120.0]})
    sample[ym] = sample[form] + 1
    assert sample[ym].astype(str).tolist() == ['2000-02', '2000-04']
    returns = pd.DataFrame({stock_code: ['000001'], ym: pd.PeriodIndex(['2000-03'], freq='M'), monthly_return: [0.10]})
    result = returns.merge(sample, on=[stock_code, ym], how='left', validate='one_to_one')
    assert result[monthly_market_value].isna().all()


def test_ch3_arithmetic_and_no_input_mutation():
    months = pd.period_range('2000-01', periods=2, freq='M')
    sample = pd.DataFrame({stock_code: np.tile(['{:06d}'.format(i) for i in range(1,101)], 2), ym: np.repeat(months,100), monthly_market_value: np.tile(np.arange(1,101,dtype=float),2), monthly_ep: np.tile(np.sin(np.arange(100)),2), monthly_excess_ret: np.repeat([0.02,-0.01],100)})
    before = sample.copy(deep=True)
    factors = get_ch3_factors(sample, cut=0.30)
    pd.testing.assert_frame_equal(sample,before)
    assert np.allclose(factors[smb],0)
    assert np.allclose(factors[vmg],0)
    assert np.allclose(factors[mkt],[0.02,-0.01])


def test_grs_invariant_to_return_units():
    rng = np.random.default_rng(1013)
    months = pd.period_range('2000-01',periods=120,freq='M')
    factors = pd.DataFrame(rng.normal(0.005,0.04,(120,2)),index=months,columns=['f1','f2'])
    returns = pd.DataFrame(factors.to_numpy() @ np.array([[1.0,0.4],[-0.2,0.8]]) + rng.normal(0,0.03,(120,2)),index=months,columns=['a','b'])
    first = grs_test(returns,factors)
    second = grs_test(100*returns,100*factors)
    assert np.isclose(first['grs'],second['grs'],rtol=1e-8)
    assert 0 <= first['p'] <= 1
    assert first['months'] == 120


def test_deciles_require_ten_observed_values():
    values = pd.Series([1.0]+[np.nan]*19)
    assert get_quantile_labels(values,10).isna().all()
    values = pd.Series(np.arange(100,dtype=float))
    labels = get_quantile_labels(values,10)
    assert set(labels.unique()) == set(range(1,11))
    assert labels.iloc[0] == 1 and labels.iloc[-1] == 10


def test_formation_universe_keeps_missing_future_return():
    sample = pd.DataFrame({stock_code:['000001','000002','000003'],form:pd.PeriodIndex(['1999-12']*3,freq='M'),ym:pd.PeriodIndex(['2000-01']*3,freq='M'),'eligible':[True]*3,monthly_market_value:[10.0,20.0,30.0]})
    realized = pd.DataFrame({stock_code:['000001','000002'],ym:pd.PeriodIndex(['2000-01']*2,freq='M'),monthly_return:[0.02,0.01],monthly_excess_ret:[0.02,0.01],'Monrfret':[0.0,0.0]})
    result = get_formation_panel(sample,realized)
    assert len(result)==3
    assert result.loc[result[stock_code].eq('000003'),monthly_return].isna().all()
    assert np.allclose(result[monthly_market_value].rank(pct=True),[1/3,2/3,1])


def test_reversal_restarts_after_absent_month():
    calendar = pd.bdate_range('2000-01-03','2000-03-31')
    ids = ['{:06d}'.format(i) for i in range(100)]
    rows = []
    for day in calendar:
        for position,code_value in enumerate(ids):
            value = (0.01 if position>=50 else -0.01) if day.month==3 else 0.0
            signal = position if day.month==1 else 100-position
            rows.append({stock_code:code_value,date:day,daily_ret:value,'daily_reversal':signal})
    data = pd.DataFrame(rows)
    caps = pd.DataFrame({stock_code:ids*2,ym:np.repeat(pd.PeriodIndex(['2000-01','2000-03'],freq='M'),100),monthly_market_value:100.0})
    result = get_daily_reversal(data,caps,False)
    march_days = int((calendar.month==3).sum())
    assert np.isclose(result.loc[pd.Period('2000-03',freq='M')],1.02**(march_days-4)-1)


def test_calendar_nw_matches_regular_hac():
    values = pd.Series(np.sin(np.arange(50)),index=pd.period_range('2000-01',periods=50,freq='M'))
    mean,t_stat,count = get_calendar_nw_mean(values,4)
    reference = sm.OLS(values.to_numpy(),np.ones((50,1))).fit(cov_type='HAC',cov_kwds={'maxlags':4})
    assert np.isclose(mean,reference.params[0])
    assert np.isclose(t_stat,reference.tvalues[0])
    assert count==50
    missing = values.drop(values.index[20])
    grid = (missing-missing.mean()).reindex(values.index)
    variance = grid.pow(2).sum()
    for lag in range(1,5):
        variance += 2*(1-lag/5)*(grid*grid.shift(lag)).sum()
    expected = missing.mean()/np.sqrt(variance/len(missing)**2)
    assert np.isclose(get_calendar_nw_mean(missing,4)[1],expected)


def test_cash_flow_uses_reported_period_balances():
    cash = pd.DataFrame({'EndDt':pd.to_datetime(['2007-06-30','2007-09-30','2001-06-30']),'EndCEBalanc':[140.0,np.nan,140.0],'BegCEBalanc':[100.0,100.0,100.0],'CENetIncr':[41.0,50.0,41.0]})
    result = get_period_cash_flow(cash)
    assert np.allclose(result['period_cash'],[40.0,50.0,20.0])
    assert 'period_cash' not in cash


def test_financial_value_waits_for_publication_and_restores_rows():
    sample = pd.DataFrame({stock_code:pd.Series(['000001','000001'],dtype='string'),'formation_end':pd.to_datetime(['2007-06-30','2006-12-31'])})
    event = pd.DataFrame({'A_StkCd':pd.Series(['000001'],dtype='string'),'InfoPubDt':pd.to_datetime(['2007-03-31']),'EndDt':pd.to_datetime(['2005-12-31']),'value':[5.0]})
    result = align_statement_values(sample,event,['value'])
    assert len(result)==2 and result.loc[0,'value']==5.0 and np.isnan(result.loc[1,'value'])


def test_dimson_sums_all_six_market_coefficients():
    rng = np.random.default_rng(313)
    calendar = pd.bdate_range('1999-01-01','2000-11-30')
    market = pd.Series(rng.normal(0,0.01,len(calendar)),index=calendar)
    lags = pd.concat([market.shift(lag).rename('lag_{}'.format(lag)) for lag in range(6)],axis=1)
    coefficients = np.array([0.7,0.2,0.1,0.0,-0.1,0.1])
    data = pd.DataFrame({stock_code:'000001',date:calendar,daily_ret:0.0002+lags.to_numpy()@coefficients})
    signal = pd.DataFrame({stock_code:['000001','000001'],form:pd.PeriodIndex(['1999-12','2000-10'],freq='M'),'formation_end':pd.to_datetime(['1999-12-31','2000-10-31'])})
    result = get_dimson_beta(data,signal,lags)
    assert np.allclose(result['monthly_beta'],coefficients.sum(),atol=1e-10)


for test in [test_weight_uses_common_valid_sample,test_monthly_alignment_skips_calendar_gaps,test_ch3_arithmetic_and_no_input_mutation,test_grs_invariant_to_return_units,test_deciles_require_ten_observed_values,test_formation_universe_keeps_missing_future_return,test_reversal_restarts_after_absent_month,test_calendar_nw_matches_regular_hac,test_cash_flow_uses_reported_period_balances,test_financial_value_waits_for_publication_and_restores_rows,test_dimson_sums_all_six_market_coefficients]:
    test()
    print('{}：通过'.format(test.__name__))
